## Step2: Fin-Bert 情感因子提取与初步映射

In [ ]:
import os
import ast
import gc
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# 1. 硬件自适应：确保 RTX 4060 发力
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"当前使用的计算设备: {device}")

# 2. 数据读取与基础预处理
data_path = '../data/01_preprocessed_data.csv'
output_path = '../data/02_finbert_features_added.csv'
df = pd.read_csv(data_path)

# 3. LLM 标签映射 (在原始数据上执行) 
sentiment_map = {'positive': 1, 'neutral': 0, 'negative': -1}
action_map = {'buy': 1, 'hold': 0, 'sell': -1}

df['sentiment_class'] = df['sentiment_class'].str.lower().map(sentiment_map).fillna(0)
df['action_class'] = df['action_class'].str.lower().map(action_map).fillna(0)
df['action_score'] = df['action_score'].astype(float).fillna(method='ffill').fillna(7.5)

# 4. 关键优化：文本预解析函数 (移出主循环)
def quick_extract(cell_value):
    if pd.isna(cell_value) or cell_value == "": return []
    try:
        parsed = ast.literal_eval(cell_value)
        # 如果是列表则展平，提取长度大于15的非链接文本 
        return [str(item[4]) if isinstance(item, list) and len(item)>4 else str(item) 
                for item in (parsed if isinstance(parsed, list) else [parsed])
                if isinstance(item, (str, list))]
    except: return []

print("正在预处理文本列 (CPU 密集型任务)...")
text_columns = ['cointelegraph', 'bitcoin_news', 'reddit']
for col in text_columns:
    df[col] = df[col].apply(quick_extract)

# 5. FinBERT 模型加载 
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

# 6. 高速推理循环 
finbert_scores = []
BATCH_SIZE = 16 # 

print("\n开始 GPU 推理...")
with torch.no_grad(): # 禁用梯度 
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="FinBERT 推理进度"):
        daily_texts = row['cointelegraph'] + row['bitcoin_news'] + row['reddit']
        
        if not daily_texts:
            finbert_scores.append(0.0)
            continue
            
        temp_scores = []
        for i in range(0, len(daily_texts), BATCH_SIZE):
            batch = daily_texts[i : i + BATCH_SIZE]
            inputs = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors='pt').to(device)
            outputs = model(**inputs)
            probs = F.softmax(outputs.logits, dim=-1) 
            # Score = P(Pos) - P(Neg)
            score = (probs[:, 0] - probs[:, 1]).mean().item()
            temp_scores.append(score)
            
        finbert_scores.append(sum(temp_scores) / len(temp_scores))

        # 每 100 行清理一次显存，减少同步开销 
        if idx % 100 == 0:
            torch.cuda.empty_cache()
            gc.collect()

# 7. 结果导出
df['finbert_sentiment_score'] = finbert_scores
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df.to_csv(output_path, index=False)
print(f"处理完成！耗时应大幅缩短。")

当前使用的计算设备: cuda


C:\Users\iocto\AppData\Local\Temp\ipykernel_15920\1877539679.py:25: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['action_score'] = df['action_score'].astype(float).fillna(method='ffill').fillna(7.5)


正在预处理文本列 (CPU 密集型任务)...

开始 GPU 推理...


FinBERT 推理进度: 100%|██████████| 2337/2337 [03:18<00:00, 11.80it/s]


处理完成！耗时应大幅缩短。


In [3]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. 绘图环境与全局字体配置
# ==========================================
# Windows 字体兼容性与负号修复
plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False    # 用来正常显示负号

# 确保图片结果保存目录存在
results_dir = '../results/'
os.makedirs(results_dir, exist_ok=True)

# ==========================================
# 2. 数据读取与准备
# ==========================================
data_path = '../data/02_finbert_features_added.csv'
print(f"正在加载数据: {data_path} ...")
df = pd.read_csv(data_path)

# 将时间戳转换为 datetime 对象并作为索引，方便时间序列绘图
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.set_index('timestamp', inplace=True)

# 确保我们需要分析的特征存在
finbert_col = ['finbert_sentiment_score']
llm_cols = ['sentiment_class', 'action_class', 'action_score']

# ==========================================
# 3. 统计表格生成 (在终端打印，供复制到 Word)
# ==========================================
print("\n" + "="*50)
print("表 3-5/3-6：FinBERT 得分与 LLM 三维特征的描述性统计表")
print("="*50)
desc_stats = df[finbert_col + llm_cols].describe().T
# 增加偏度(skew)和峰度(kurt)，让统计表在论文里更丰满
desc_stats['skewness'] = df[finbert_col + llm_cols].skew()
desc_stats['kurtosis'] = df[finbert_col + llm_cols].kurt()
print(desc_stats.round(4))

print("\n" + "="*50)
print("表 3-7：LLM 原始三特征（映射后）的相关系数矩阵")
print("="*50)
corr_matrix = df[llm_cols].corr()
print(corr_matrix.round(4))

# ==========================================
# 4. 图 3-3：FinBERT 情感得分（7日移动平均）时间序列图
# ==========================================
print("\n正在绘制 图 3-3：FinBERT 情感得分时间序列图...")
# 计算 7 日移动平均
df['finbert_ma7'] = df['finbert_sentiment_score'].rolling(window=7, min_periods=1).mean()

fig1, ax1 = plt.subplots(figsize=(12, 5))
# 绘制 7 日均值线
ax1.plot(df.index, df['finbert_ma7'], color='#1f77b4', linewidth=1.5, label='FinBERT 情感得分 (7日均线)')
# 添加零轴辅助线以区分正负情绪
ax1.axhline(0, color='red', linestyle='--', linewidth=1, alpha=0.7)

# 遵循规范：无 plt.title()，规范化中英文轴标签
ax1.set_xlabel('日期 (Date)', fontsize=12)
ax1.set_ylabel('FinBERT 情感得分', fontsize=12)
ax1.legend(loc='upper right', frameon=True)

# 调整布局并保存
plt.tight_layout()
fig3_3_path = os.path.join(results_dir, '3_3_finbert_ma7.png')
plt.savefig(fig3_3_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"-> 图 3-3 已保存至: {fig3_3_path}")


# ==========================================
# 4.5 图 3-4：FinBERT 情感得分分布直方图
# ==========================================
print("\n正在绘制 图 3-4：FinBERT 情感得分分布直方图...")
fig_dist, ax_dist = plt.subplots(figsize=(8, 5))

# 使用 seaborn 绘制直方图并叠加 KDE 核密度曲线，观察分布形态
sns.histplot(df['finbert_sentiment_score'], bins=50, kde=True, color='#2ca02c', ax=ax_dist)

# 添加零轴辅助线，直观看出正负情感的分布偏移
ax_dist.axvline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.7)

# 遵循规范：无 plt.title()，规范化中英文轴标签
ax_dist.set_xlabel('FinBERT 情感得分 (Sentiment Score)', fontsize=12)
ax_dist.set_ylabel('频数 (Days)', fontsize=12)

# 调整布局并保存
plt.tight_layout()
fig3_4_path = os.path.join(results_dir, '3_4_finbert_distribution.png')
plt.savefig(fig3_4_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"-> 图 3-4 已保存至: {fig3_4_path}")



# ==========================================
# 5. 图 3-5：sentiment_class 与 action_class 的交叉分布热力图
# ==========================================
print("正在绘制 图 3-5：LLM 情感与行动建议交叉分布热力图...")
# 使用 crosstab 生成交叉频数表
cross_tab = pd.crosstab(df['sentiment_class'], df['action_class'])

# 由于类别是 -1, 0, 1，可以显式重命名索引和列名，使热力图标签更易读（可选）
cross_tab.index = ['负面 (-1)', '中性 (0)', '正面 (1)']
cross_tab.columns = ['卖出 (-1)', '持有 (0)', '买入 (1)']

fig2, ax2 = plt.subplots(figsize=(8, 6))
# 绘制热力图，annot=True 显示具体数值，fmt='d' 整数格式，使用 Blues 色系
sns.heatmap(cross_tab, annot=True, fmt='d', cmap='Blues', ax=ax2, cbar_kws={'label': '频数 (天)'})

# 遵循规范：无 plt.title()
ax2.set_ylabel('情感类别 (sentiment_class)', fontsize=12)
ax2.set_xlabel('行动建议 (action_class)', fontsize=12)

# 调整布局并保存
plt.tight_layout()
fig3_5_path = os.path.join(results_dir, '3_5_sentiment_action_heatmap.png')
plt.savefig(fig3_5_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"-> 图 3-5 已保存至: {fig3_5_path}")

print("\n所有图表和表格输出任务执行完毕！")

正在加载数据: ../data/02_finbert_features_added.csv ...

表 3-5/3-6：FinBERT 得分与 LLM 三维特征的描述性统计表
                          count    mean     std     min     25%     50%  \
finbert_sentiment_score  2337.0 -0.0596  0.0313 -0.2772 -0.0768 -0.0591   
sentiment_class          2337.0  0.2593  0.7457 -1.0000  0.0000  0.0000   
action_class             2337.0  0.1853  0.4802 -1.0000  0.0000  0.0000   
action_score             2337.0  6.8169  1.0282  5.0000  6.0000  6.0000   

                            75%      max  skewness  kurtosis  
finbert_sentiment_score -0.0414   0.0608   -0.5527    2.3827  
sentiment_class          1.0000   1.0000   -0.4600   -1.0837  
action_class             0.0000   1.0000    0.4590    0.3632  
action_score             8.0000  10.0000    0.6225   -0.6763  

表 3-7：LLM 原始三特征（映射后）的相关系数矩阵
                 sentiment_class  action_class  action_score
sentiment_class           1.0000        0.6033        0.5946
action_class              0.6033        1.0000        0.5065
action_s